In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_bronze = spark.read.table("top_headlines")

print("Bronze row count:", df_bronze.count())
df_bronze.printSchema()
df_bronze.show(3, truncate=80)

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 3, Finished, Available, Finished, False)

Bronze row count: 18
root
 |-- source_name: string (nullable = true)
 |-- author: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- url: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- content: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

+----------------------------+----------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------+--------------------------------------------------------------------------------+--------------------------+
|                 source_name|          author|                                                                           title|                                                                     description|                   

In [2]:
from pyspark.sql.functions import (
    col, lower, when, regexp_extract, to_timestamp,
    trim, length, row_number
)
from pyspark.sql.window import Window

# Start from bronze
df = spark.read.table("top_headlines")

# --- 1. Filter junk rows ---
df = df.filter(
    col("title").isNotNull() &
    (col("title") != "[Removed]") &        # News API marks deleted articles like this
    (length(trim(col("title"))) > 5)       # drop nonsense like ""
)

# --- 2. Parse published_at into a real timestamp ---
df = df.withColumn(
    "published_ts",
    to_timestamp(col("published_at"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# --- 3. Extract domain from URL (apnews.com from https://apnews.com/article/...) ---
df = df.withColumn(
    "domain",
    regexp_extract(col("url"), r"https?://(?:www\.)?([^/]+)", 1)
)

# --- 4. Categorize by keywords in title (lowercase to match) ---
title_lc = lower(col("title"))
df = df.withColumn(
    "category",
    when(title_lc.rlike(r"\b(ai|tech|software|google|apple|microsoft|samsung|iphone|chip|crypto)\b"), "Tech")
    .when(title_lc.rlike(r"\b(trump|biden|congress|senate|election|gop|democrat|republican|white house)\b"), "Politics")
    .when(title_lc.rlike(r"\b(stock|market|economy|inflation|fed|earnings|wall street|jobs|layoff)\b"), "Business")
    .when(title_lc.rlike(r"\b(nfl|nba|mlb|nhl|soccer|olympics|world cup|playoff|championship)\b"), "Sports")
    .when(title_lc.rlike(r"\b(war|israel|gaza|ukraine|russia|iran|china|nato)\b"), "World")
    .otherwise("Other")
)

# --- 5. Deduplicate by url — keep most recent ingestion ---
window = Window.partitionBy("url").orderBy(col("ingested_at").desc())
df = df.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop("rn")

# --- 6. Final column selection — clean schema ---
df_silver = df.select(
    "title",
    "source_name",
    "domain",
    "category",
    "author",
    "description",
    "url",
    "published_ts",
    "ingested_at"
)

print("Silver row count:", df_silver.count())
df_silver.show(5, truncate=70)

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 4, Finished, Available, Finished, False)

Silver row count: 18
+----------------------------------------------------------------------+----------------+------------------+--------+------------------------------------------------+----------------------------------------------------------------------+----------------------------------------------------------------------+-------------------+--------------------------+
|                                                                 title|     source_name|            domain|category|                                          author|                                                           description|                                                                   url|       published_ts|               ingested_at|
+----------------------------------------------------------------------+----------------+------------------+--------+------------------------------------------------+----------------------------------------------------------------------+------------------------------

In [3]:
from pyspark.sql.functions import count

df_silver.groupBy("category") \
  .agg(count("*").alias("article_count")) \
  .orderBy("article_count", ascending=False) \
  .show()

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 5, Finished, Available, Finished, False)

+--------+-------------+
|category|article_count|
+--------+-------------+
|   Other|           11|
|Politics|            3|
|  Sports|            2|
|    Tech|            2|
+--------+-------------+



In [4]:
df_silver.write.mode("overwrite").saveAsTable("top_headlines_silver")

print("✅ Silver table saved")
print("Rows:", spark.read.table("top_headlines_silver").count())

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 6, Finished, Available, Finished, False)

✅ Silver table saved
Rows: 18


In [6]:
spark.sql("""
    SELECT category, domain, source_name, title 
    FROM top_headlines_silver 
    ORDER BY category, source_name
""").show(20, truncate=70)

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 8, Finished, Available, Finished, False)

+--------+------------------+----------------------------+----------------------------------------------------------------------+
|category|            domain|                 source_name|                                                                 title|
+--------+------------------+----------------------------+----------------------------------------------------------------------+
|   Other|       abcnews.com|                 Abcnews.com|Could 8,500 daily steps help keep weight off? - ABC News - Breaking...|
|   Other|   arstechnica.com|                Ars Technica|Linux bitten by second severe vulnerability in as many weeks - Ars ...|
|   Other|        apnews.com|            Associated Press|Heat stroke is suspected among 6 found dead in a shipping container...|
|   Other|           ajc.com|Atlanta Journal Constitution|Andrew Morse to step down, Paul Curran named AJC president and publ...|
|   Other|         axios.com|                       Axios|Byron Allen strikes deal to buy 

In [7]:
spark.sql("""
    SELECT domain, COUNT(*) AS articles 
    FROM top_headlines_silver 
    GROUP BY domain 
    ORDER BY articles DESC
""").show()

StatementMeta(, efd7b7ef-66dd-4da2-8a5b-90cae0b68365, 9, Finished, Available, Finished, False)

+------------------+--------+
|            domain|articles|
+------------------+--------+
|    techcrunch.com|       2|
|spaceflightnow.com|       1|
|          wral.com|       1|
|      politico.com|       1|
|   news.google.com|       1|
|           wsj.com|       1|
|   arstechnica.com|       1|
|       raiders.com|       1|
|          espn.com|       1|
|           ajc.com|       1|
|           cnn.com|       1|
|       foxnews.com|       1|
|           npr.org|       1|
|       abcnews.com|       1|
|         axios.com|       1|
|       thehill.com|       1|
|        apnews.com|       1|
+------------------+--------+

